In [1]:
#pip install seaborn

In [2]:
#pip install matplotlib

In [3]:
#pip install openpyxl

In [4]:
# Import bibliotecas

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
df = pd.read_csv('history.xlsx', sep=';', encoding='utf-8-sig',
                 na_values=['null', 'NULL', 'unknown'])
df_bronze = df.copy()
df.head()


   customerID  ... MonthlyIncome
0  4578-PHJYZ  ...        6532.0
1  6289-CPNLD  ...        7634.0
2  2682-KEVRP  ...        3628.0
3  5697-GOMBF  ...        7851.0
4  9717-QEBGU  ...        1691.0

[5 rows x 23 columns]

In [6]:
# Verificando a dimensão do conjunto de dados
df.shape

(5912, 23)

In [7]:
# Informações dos tipos de variáveis
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5912 entries, 0 to 5911
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        5773 non-null   str    
 1   gender            5768 non-null   str    
 2   SeniorCitizen     5747 non-null   str    
 3   Partner           5687 non-null   str    
 4   Dependents        5644 non-null   str    
 5   tenure            5618 non-null   str    
 6   PhoneService      5635 non-null   str    
 7   MultipleLines     5634 non-null   str    
 8   InternetService   5634 non-null   str    
 9   OnlineSecurity    5634 non-null   str    
 10  OnlineBackup      5634 non-null   str    
 11  DeviceProtection  5634 non-null   str    
 12  TechSupport       5634 non-null   str    
 13  StreamingTV       5634 non-null   str    
 14  StreamingMovies   5634 non-null   str    
 15  Contract          5634 non-null   str    
 16  PaperlessBilling  5634 non-null   str    
 17  Paymen

In [8]:
# Verificando valores nulos
df.isnull().sum()

customerID          139
gender              144
SeniorCitizen       165
Partner             225
Dependents          268
tenure              294
PhoneService        277
MultipleLines       278
InternetService     278
OnlineSecurity      278
OnlineBackup        278
DeviceProtection    278
TechSupport         278
StreamingTV         278
StreamingMovies     278
Contract            278
PaperlessBilling    278
PaymentMethod       278
MonthlyCharges      278
TotalCharges        556
Churn               278
CustomerFeedback    278
MonthlyIncome       363
dtype: int64

In [9]:
# Verificar a quantidade total de valores nulos
df.isnull().sum().sum()

np.int64(6323)

#### 1. Limpeza


In [10]:
# Limpar espaços nos textos e transformar campos vazios em nulos
for coluna in df.select_dtypes(include=['object', 'string']).columns:
    df[coluna] = df[coluna].str.strip()

df = df.replace(['', 'null', 'NULL', 'unknown'], pd.NA)
df['customerID'] = df['customerID'].str.upper()

# ID válido: quatro números, hífen e cinco letras (exemplo: 4578-PHJYZ)
id_valido = df['customerID'].str.fullmatch(r'[0-9]{4}-[A-Z]{5}', na=False)

df_linhas_invalidas = df[~id_valido].copy()
df = df[id_valido].copy()

# Remover somente linhas totalmente iguais
df = df.drop_duplicates().reset_index(drop=True)
print('Clientes:', len(df))
print('Linhas separadas para conferir:', len(df_linhas_invalidas))

Clientes: 5634
Linhas separadas para conferir: 278


#### 2. Tradução
Traduzimos os valores e os nomes das colunas antes da tipagem. Os feedbacks mantêm o texto original. Na próxima etapa, as respostas Sim/Não serão convertidas em booleanos.


In [31]:
# Traduções usadas em mais de uma coluna
sim_nao = {
    'yes': 'Sim',
    'no': 'Não'
}

servico_internet = {
    'yes': 'Sim',
    'no': 'Não',
    'no internet service': 'Sem serviço de internet'
}

# Tradução de cada coluna
traducao_valores = {
    'gender': {
        'male': 'Masculino',
        'female': 'Feminino'
    },
    'Partner': sim_nao,
    'Dependents': sim_nao,
    'PhoneService': sim_nao,
    'PaperlessBilling': sim_nao,
    'Churn': sim_nao,

    'MultipleLines': {
        'yes': 'Sim',
        'no': 'Não',
        'no phone service': 'Sem serviço de telefonia'
    },
    'InternetService': {
        'dsl': 'DSL',
        'fiber optic': 'Fibra óptica',
        'no': 'Sem serviço de internet'
    },
    'OnlineSecurity': servico_internet,
    'OnlineBackup': servico_internet,
    'DeviceProtection': servico_internet,
    'TechSupport': servico_internet,
    'StreamingTV': servico_internet,
    'StreamingMovies': servico_internet,

    'Contract': {
        'month-to-month': 'Mensal',
        'one year': 'Um ano',
        'two year': 'Dois anos'
    },
    'PaymentMethod': {
        'electronic check': 'Cheque eletrônico',
        'mailed check': 'Cheque pelo correio',
        'bank transfer (automatic)': 'Transferência bancária automática',
        'credit card (automatic)': 'Cartão de crédito automático'
    }
}

# Colocar os textos em minúsculas e aplicar as traduções
for coluna, mapa in traducao_valores.items():
    df[coluna] = df[coluna].str.lower().replace(mapa)

# Conferir o resultado
df[['gender', 'InternetService', 'Contract', 'PaymentMethod', 'Churn']].head()


,gender,InternetService,Contract,PaymentMethod,Churn
0,Masculino,DSL,Um ano,Cheque eletrônico,Não
1,Masculino,DSL,Um ano,Cheque pelo correio,Sim
2,Feminino,Sem serviço de internet,Um ano,Cheque pelo correio,Não
3,Feminino,DSL,Mensal,Cheque eletrônico,Não
4,Masculino,Fibra óptica,Mensal,Cheque eletrônico,Não


In [32]:
# Traduzir as colunas de acordo com a descrição da sua base
traducao_colunas = {
    'customerID': 'ID_Cliente', 
    'gender': 'Genero',
    'SeniorCitizen': 'Idoso',
    'Partner': 'Conjuge', 
    'Dependents': 'Dependentes', 
    'tenure': 'Tempo_Assinatura',
    'PhoneService': 'Servico_Telefonia', 
    'MultipleLines': 'Varias_Linhas',
    'InternetService': 'Servico_Internet', 
    'OnlineSecurity': 'Seguranca_Online',
    'OnlineBackup': 'Backup_Online', 
    'DeviceProtection': 'Protecao_Dispositivos',
    'TechSupport': 'Suporte_Tecnico', 
    'StreamingTV': 'Streaming_TV',
    'StreamingMovies': 'Streaming_Filmes', 
    'Contract': 'Contrato',
    'PaperlessBilling': 'Faturamento_Eletronico', 
    'PaymentMethod': 'Forma_Pagamento',
    'MonthlyCharges': 'Valores_Mensais', 
    'TotalCharges': 'Valores_Totais',
    'Churn': 'Status_Cancelamento', 
    'CustomerFeedback': 'Feedback_Cliente',
    'MonthlyIncome': 'Renda_Mensal'
}
df = df.rename(columns=traducao_colunas)


df.head()

,ID_Cliente,Genero,Idoso,Conjuge,Dependentes,Tempo_Assinatura,Servico_Telefonia,Varias_Linhas,Servico_Internet,Seguranca_Online,Backup_Online,Protecao_Dispositivos,Suporte_Tecnico,Streaming_TV,Streaming_Filmes,Contrato,Faturamento_Eletronico,Forma_Pagamento,Valores_Mensais,Valores_Totais,Status_Cancelamento,Feedback_Cliente,Renda_Mensal
0,4578-PHJYZ,Masculino,0,Sim,Sim,52,Sim,Não,DSL,Não,Sim,Sim,Sim,Sim,Não,Um ano,Sim,Cheque eletrônico,68.75,3482.85,Não,I have been a customer with this internet prov...,6532
1,6289-CPNLD,Masculino,0,Sim,Sim,33,Sim,Não,DSL,Não,Não,Sim,Sim,Sim,Sim,Um ano,Sim,Cheque pelo correio,73.9,2405.05,Sim,I've been a customer with this company for ove...,7634
2,2682-KEVRP,Feminino,1,Não,Não,22,Sim,Não,Sem serviço de internet,Sem serviço de internet,Sem serviço de internet,Sem serviço de internet,Sem serviço de internet,Sem serviço de internet,Sem serviço de internet,Um ano,Sim,Cheque pelo correio,20.05,417.0,Não,I have been a customer with this company for 2...,3628
3,5697-GOMBF,Feminino,1,Sim,Sim,28,Não,Sem serviço de telefonia,DSL,Não,Não,Não,Não,Sim,Não,Mensal,Sim,Cheque eletrônico,35.9,973.65,Não,I have been a customer with this internet prov...,7851
4,9717-QEBGU,Masculino,0,Não,Não,2,Sim,Não,Fibra óptica,Não,Não,Não,Não,Não,Não,Mensal,Sim,Cheque eletrônico,70.3,144.0,Não,I have been using the fiber optic internet ser...,1691


#### 3. Tipagem — tudo nesta célula
Tempo de assinatura fica inteiro; cobranças e renda ficam decimais. Campos de Sim/Não ficam booleanos: `True` significa Sim e `False` significa Não. Para Idoso, 1 vira True e 0 vira False.

Gênero e os tipos de serviços, contrato e pagamento ficam categóricos. Dependentes também é booleano; backup, proteção e suporte são categorias porque incluem a opção sem serviço de internet.

Os tipos `boolean`, `Int64` e `Float64` aceitam nulos. Os valores ausentes permanecem nulos, inclusive os tempos marcados como unknown na origem.


In [ ]:
# Números: tempo em meses, cobranças e renda
colunas_numericas = ['Tempo_Assinatura', 'Idoso', 'Valores_Mensais', 'Valores_Totais', 'Renda_Mensal']
df[colunas_numericas] = df[colunas_numericas].apply(pd.to_numeric, errors='raise')

df = df.astype({
    'Tempo_Assinatura': 'Int64',
    'Valores_Mensais': 'Float64',
    'Valores_Totais': 'Float64',
    'Renda_Mensal': 'Float64'
})

df['Idoso'] = df['Idoso'].astype('boolean')

colunas_booleanas = ['Conjuge', 'Dependentes', 'Servico_Telefonia',
                     'Faturamento_Eletronico', 'Status_Cancelamento']
for coluna in colunas_booleanas:
    df[coluna] = df[coluna].replace({'Sim': True, 'Não': False}).astype('boolean')

colunas_categoricas = ['Genero', 'Servico_Internet', 'Varias_Linhas',
                      'Seguranca_Online', 'Backup_Online', 'Protecao_Dispositivos',
                      'Suporte_Tecnico', 'Streaming_TV', 'Streaming_Filmes',
                      'Contrato', 'Forma_Pagamento']
for coluna in colunas_categoricas:
    df[coluna] = df[coluna].astype('category')

df.info()


<class 'pandas.DataFrame'>
RangeIndex: 5634 entries, 0 to 5633
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   ID_Cliente              5634 non-null   str     
 1   Genero                  5634 non-null   category
 2   Idoso                   5634 non-null   boolean 
 3   Conjuge                 5634 non-null   boolean 
 4   Dependentes             5634 non-null   boolean 
 5   Tempo_Assinatura        5615 non-null   Int64   
 6   Servico_Telefonia       5634 non-null   boolean 
 7   Varias_Linhas           5634 non-null   category
 8   Servico_Internet        5634 non-null   category
 9   Seguranca_Online        5634 non-null   category
 10  Backup_Online           5634 non-null   category
 11  Protecao_Dispositivos   5634 non-null   category
 12  Suporte_Tecnico         5634 non-null   category
 13  Streaming_TV            5634 non-null   category
 14  Streaming_Filmes        5634 non-nu

In [14]:

df_silver = df.copy()

tabela_nulos = pd.DataFrame({
    'Total Nulos': df_silver.isnull().sum(),
    'Porcentagem (%)': (df_silver.isnull().mean() * 100).round(2)
})
tabela_nulos


                        Total Nulos  Porcentagem (%)
ID_Cliente                        0             0.00
Genero                            0             0.00
Idoso                             0             0.00
Conjuge                           0             0.00
Dependentes                       0             0.00
Tempo_Assinatura                 19             0.34
Servico_Telefonia                 0             0.00
Varias_Linhas                     0             0.00
Servico_Internet                  0             0.00
Seguranca_Online                  0             0.00
Backup_Online                     0             0.00
Protecao_Dispositivos             0             0.00
Suporte_Tecnico                   0             0.00
Streaming_TV                      0             0.00
Streaming_Filmes                  0             0.00
Contrato                          0             0.00
Faturamento_Eletronico            0             0.00
Forma_Pagamento                   0           

In [15]:
# Conferir os dados tratados e traduzidos
df_silver.head(10)

   ID_Cliente  ... Renda_Mensal
0  4578-PHJYZ  ...       6532.0
1  6289-CPNLD  ...       7634.0
2  2682-KEVRP  ...       3628.0
3  5697-GOMBF  ...       7851.0
4  9717-QEBGU  ...       1691.0
5  6797-LNAQX  ...       5487.0
6  3050-GBUSH  ...       6382.0
7  8020-BWHYL  ...        353.0
8  6752-APNJL  ...       2436.0
9  1760-CAZHT  ...       7049.0

[10 rows x 23 columns]